# Clustering threshold comparison

Compare classifier performance across sequence identity thresholds. [Results by threshold](../../results/data_analysis/task1_clustering/)


In [ ]:
import sys
sys.path.append('..')

## Analyzing and comparing the performance of differently clustered datasets for Task 1

### 1. Import relevant libraries

In [ ]:
import pandas as pd
from src.preprocessing import batch_encode_and_flatten, filter_lengths, undersample
from src.utils import class_distribution, length_distribution, df_to_fasta, df_to_tsv
import numpy as np
from protlearn import features
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch
from pathlib import Path

### 2. Load datasets

In [ ]:
df30 = pd.read_csv('../data/clustered/exp_proteins_c30_enz.tsv', sep='\t')
df50 = pd.read_csv('../data/clustered/exp_proteins_c50_enz.tsv', sep='\t')
df70 = pd.read_csv('../data/clustered/exp_proteins_c70_enz.tsv', sep='\t')
df90 = pd.read_csv('../data/clustered/exp_proteins_c90_enz.tsv', sep='\t')
df100 = pd.read_csv('../data/clustered/exp_proteins_c100_enz.tsv', sep='\t')

### 3. Curate data

#### 3.1 Filter lengths

In [ ]:
# Define Percentiles
p1 = 25
p2 = 75

df30 = filter_lengths(df30, p1, p2)
df50 = filter_lengths(df50, p1, p2)
df70 = filter_lengths(df70, p1, p2)
df90 = filter_lengths(df90, p1, p2)
df100 = filter_lengths(df100, p1, p2)

#### 3.2 Undersample

In [ ]:
undersampled = True
df30 = undersample(df30)
df50 = undersample(df50)
df70 = undersample(df70)
df90 = undersample(df90)
df100 = undersample(df100)

### 4. Initial dataset statistics

#### 4.1 Class distributions

In [ ]:
class_distribution(df30, 'df30')
class_distribution(df50, 'df50')
class_distribution(df70, 'df70')
class_distribution(df90, 'df90')
class_distribution(df100, 'df100')

#### 4.2 Length distributions

In [ ]:
length_distribution(df30, 'df30')
length_distribution(df50, 'df50')
length_distribution(df70, 'df70')
length_distribution(df90, 'df90')
length_distribution(df100, 'df100')

#### 4.3 Dataset Sizes

In [ ]:
len_dict = {'df30' : len(df30), 'df50' : len(df50), 'df70' : len(df70), 'df90' : len(df90), 'df100' : len(df100)}
X = list(len_dict.keys())
y = list(len_dict.values())
bars = plt.bar(X, y)
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height, str(height), ha='center', va='bottom')
plt.show()

### 5. Extract sequences and labels

In [ ]:
X_30 = list(df30['Sequence'])
y_30 = list(df30['Enzyme'])

X_50 = list(df50['Sequence'])
y_50 = list(df50['Enzyme'])

X_70 = list(df70['Sequence'])
y_70 = list(df70['Enzyme'])

X_90 = list(df90['Sequence'])
y_90 = list(df90['Enzyme'])

X_100 = list(df100['Sequence'])
y_100 = list(df100['Enzyme'])

In [ ]:
class Task1Dataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
dataset30 = Task1Dataset(X_30, y_30)
dataset50 = Task1Dataset(X_50, y_50)
dataset70 = Task1Dataset(X_70, y_70)
dataset90 = Task1Dataset(X_90, y_90)
dataset100 = Task1Dataset(X_100, y_100)

### 6. Feature extraction

#### 6.1 One-Hot-Encoding

In [ ]:
X_30_ohe = batch_encode_and_flatten(X_30)
X_50_ohe = batch_encode_and_flatten(X_50)
X_70_ohe = batch_encode_and_flatten(X_70)
X_90_ohe = batch_encode_and_flatten(X_90)
X_100_ohe = batch_encode_and_flatten(X_100)

#### 6.2 AAIndex Encoding

In [ ]:
X_30_aaindex = features.aaindex1(X_30, standardize='zscore')[0]
X_50_aaindex = features.aaindex1(X_50, standardize='zscore')[0]
X_70_aaindex = features.aaindex1(X_70, standardize='zscore')[0]
X_90_aaindex = features.aaindex1(X_90, standardize='zscore')[0]
X_100_aaindex = features.aaindex1(X_100, standardize='zscore')[0]

#### 6.4 Construct PyTorch Datasets

In [ ]:
class Task1Dataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(dim=1)
    
    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
dataset30 = Task1Dataset(X_30_aaindex, y_30)
dataset50 = Task1Dataset(X_50_aaindex, y_50)
dataset70 = Task1Dataset(X_70_aaindex, y_70)
dataset90 = Task1Dataset(X_90_aaindex, y_90)
dataset100 = Task1Dataset(X_100_aaindex, y_100)

### 7. Model Selection

#### 7.1 Random Forest

In [ ]:
rf = RandomForestClassifier()

#### 7.2 Neural Network

##### 7.2.1 Model 1

In [ ]:
class Task1ModelAAIndex1(nn.Module):
    def __init__(self):
        super().__init__()
        self.le_relu = nn.LeakyReLU()
        self.hidden_layer_1 = nn.Linear(553, 369, dtype=torch.float32)
        self.dropout = nn.Dropout(0.5)
        self.hidden_layer_2 = nn.Linear(369, 128, dtype=torch.float32)
        self.output_layer = nn.Linear(128, 1, dtype=torch.float32)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        h1 = self.le_relu(self.hidden_layer_1(x))
        d1 = self.dropout(h1)
        h2 = self.le_relu(self.hidden_layer_2(d1))
        return self.sigmoid(self.output_layer(h2))

### 8. Cross Validation

#### 8.1 Random Forest

In [ ]:
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=523)
cv_30 = cross_val_score(rf, X_30_ohe, y_30, cv=skfold, scoring='accuracy')
cv_50 = cross_val_score(rf, X_50_ohe, y_50, cv=skfold, scoring='accuracy')
cv_70 = cross_val_score(rf, X_70_ohe, y_70, cv=skfold, scoring='accuracy')
cv_90 = cross_val_score(rf, X_90_ohe, y_90, cv=skfold, scoring='accuracy')
cv_100 = cross_val_score(rf, X_100_ohe, y_100, cv=skfold, scoring='accuracy')

In [ ]:
results = pd.DataFrame([
    {"threshold": 30, "mean_acc": cv_30.mean(), "std_acc": cv_30.std(), 'dataset_size': len(df30)},
    {"threshold": 50, "mean_acc": cv_50.mean(), "std_acc": cv_50.std(), 'dataset_size': len(df50)},
    {"threshold": 70, "mean_acc": cv_70.mean(), "std_acc": cv_70.std(), 'dataset_size': len(df70)},
    {"threshold": 90, "mean_acc": cv_90.mean(), "std_acc": cv_90.std(), 'dataset_size': len(df90)},
    {"threshold": 100, "mean_acc": cv_100.mean(), "std_acc": cv_100.std(), 'dataset_size': len(df100)}
])
results.to_csv('../results/performance_eval/threshold_comparison/cthresh_comp_cv_ohe_rf_flens10-90_usamp.tsv', sep='\t')

#### 8.2 Neural Network

##### 8.2.1 SGD, BCELoss, lr = 0.01, epochs = 100

In [ ]:
criterion = nn.BCELoss()

In [ ]:
cv_30 = cv.cross_validate(Task1ModelAAIndex1, dataset30, optim.SGD, criterion, 5, 32, 0.02, 100)
cv_50 = cv.cross_validate(Task1ModelAAIndex1, dataset50, optim.SGD, criterion, 5, 32, 0.02, 100)
cv_70 = cv.cross_validate(Task1ModelAAIndex1, dataset70, optim.SGD, criterion, 5, 32, 0.02, 100)
cv_90 = cv.cross_validate(Task1ModelAAIndex1, dataset90, optim.SGD, criterion, 5, 32, 0.02, 100)
cv_100 = cv.cross_validate(Task1ModelAAIndex1, dataset100, optim.SGD, criterion, 5, 32, 0.02, 100)

In [ ]:
results = pd.DataFrame([
    {"threshold": 30, "mean_acc": cv_30[:, 0].mean(), "std_acc": cv_30[:, 0].std(), 'dataset_size': len(df30)},
    {"threshold": 50, "mean_acc": cv_50[:, 0].mean(), "std_acc": cv_50[:, 0].std(), 'dataset_size': len(df50)},
    {"threshold": 70, "mean_acc": cv_70[:, 0].mean(), "std_acc": cv_70[:, 0].std(), 'dataset_size': len(df70)},
    {"threshold": 90, "mean_acc": cv_90[:, 0].mean(), "std_acc": cv_90[:, 0].std(), 'dataset_size': len(df90)},
    {"threshold": 100, "mean_acc": cv_100[:, 0].mean(), "std_acc": cv_100[:, 0].std(), 'dataset_size': len(df100)}
])
results.to_csv('performance_eval/eval_results/cluster_thresh_aaindex_nn_model1_SGD_BCELoss_lr002_e100_cv_comparison_filter_lens_25-75_undersampled_.tsv', sep='\t')

#### 9. Optional: Save datasets to preprocessed/

In [ ]:
temp = 'usamp' if undersampled else ''
df_to_fasta(df30, f'../data/preprocessed/exp_proteins_c30_flens{p1}-{p2}_{temp}_enz.fasta')
df_to_fasta(df50, f'../data/preprocessed/exp_proteins_c50_flens{p1}-{p2}_{temp}_enz.fasta')
df_to_fasta(df70, f'../data/preprocessed/exp_proteins_c70_flens{p1}-{p2}_{temp}_enz.fasta')
df_to_fasta(df90, f'../data/preprocessed/exp_proteins_c90_flens{p1}-{p2}_{temp}_enz.fasta')
df_to_fasta(df100, f'../data/preprocessed/exp_proteins_c100_flens{p1}-{p2}_{temp}_enz.fasta')

df_to_tsv(df30, f'../data/preprocessed/exp_proteins_c30_flens{p1}-{p2}_{temp}_enz.tsv')
df_to_tsv(df50, f'../data/preprocessed/exp_proteins_c50_flens{p1}-{p2}_{temp}_enz.tsv')
df_to_tsv(df70, f'../data/preprocessed/exp_proteins_c70_flens{p1}-{p2}_{temp}_enz.tsv')
df_to_tsv(df90, f'../data/preprocessed/exp_proteins_c90_flens{p1}-{p2}_{temp}_enz.tsv')
df_to_tsv(df100, f'../data/preprocessed/exp_proteins_c100_flens{p1}-{p2}_{temp}_enz.tsv')